# dataloader-pin-memory-workers — ex2: DataLoader with shuffle=True + seeded worker_init_fn — verify batch shapes and full-epoch coverage

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `dataloader-pin-memory-workers`. Running the final beacon cell reports progress against the `PyTorch: DataLoader pin_memory + workers` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: DataLoader pin_memory + workers` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dataloader-pin-memory-workers`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dataloader-pin-memory-workers"
DD_SUBTOPIC = "PyTorch: DataLoader pin_memory + workers"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## DataLoader + `worker_init_fn` + shuffle — batch shape stays correct

Ex1 built a DataLoader with `num_workers` and `pin_memory` configured. The deepening move shows that under `shuffle=True` with multiple workers, batch shapes (and batch counts) remain correct as long as the main-process seed is set AND each worker gets a deterministic seed via `worker_init_fn`.

```python
def worker_init_fn(worker_id):
    base = torch.initial_seed() % (2**32)
    np.random.seed(base + worker_id)
    random.seed(base + worker_id)
```

**Why this matters.** `torch.initial_seed()` inside a worker returns the per-worker base seed PyTorch assigns. Without re-seeding numpy and Python `random`, every worker draws the same numpy/random stream — a silent source of duplicate augmentations in data-augmentation pipelines.

**Shape invariant.** Regardless of shuffle / workers / pin_memory, the DataLoader still yields `(batch_size, *item_shape)` tensors and the total number of items emitted across one epoch equals `len(dataset)`. `pin_memory=False` on CPU; the shape contract is unchanged.

**CPU-only is fine.** `pin_memory=True` requires CUDA at runtime. In a CPU test we pass `pin_memory=False` and still exercise the rest of the config (workers, shuffle, seeded init).

### Exercise 2 — DataLoader with shuffle=True + seeded worker_init_fn — verify batch shapes and full-epoch coverage

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `worker_init_fn` + `torch.initial_seed()` to seed each worker's numpy/random RNG deterministically, and verify that under `shuffle=True` the resulting DataLoader still yields `(batch_size, *item_shape)` batches and covers every dataset item in one epoch.
> Keywords: dataloader, worker_init_fn, shuffle, batch-shape
> ```

**KCs targeted:** `worker-init-fn-seeded-rng`, `shuffle-preserves-batch-shape`

Implement `ex2_make_seeded_shuffle_dataloader(dataset, batch_size, num_workers)`. Returns a `DataLoader` configured for:

1. `batch_size` items per batch.
2. `num_workers` worker processes (may be 0 for in-main).
3. `shuffle=True`.
4. `pin_memory=False` (CPU-only test environment).
5. `drop_last=False` — keep the last partial batch.
6. `worker_init_fn` that seeds numpy AND Python `random` per worker via `torch.initial_seed()`:
```python
def worker_init_fn(worker_id):
    import numpy as np, random
    base = torch.initial_seed() % (2**32)
    np.random.seed(base + worker_id)
    random.seed(base + worker_id)
```

Return the configured DataLoader.

**The test then verifies:**
- batch shape `(B, *item_shape)`, including last batch when `len(dataset) % batch_size != 0`.
- total items across one epoch == `len(dataset)`.
- two epochs over the same loader yield DIFFERENT orderings (shuffle works) but the same total item count.

**Use `from torch.utils.data import DataLoader, TensorDataset`.**

In [ ]:
from torch.utils.data import DataLoader
import torch as _t
import numpy as _np
import random as _random

def _worker_init_fn(worker_id):
    base = _t.initial_seed() % (2 ** 32)
    _np.random.seed(base + worker_id)
    _random.seed(base + worker_id)

def ex2_make_seeded_shuffle_dataloader(dataset, batch_size, num_workers):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=num_workers,
        shuffle=True,
        pin_memory=False,
        drop_last=False,
        worker_init_fn=_worker_init_fn,
    )


<details><summary>Solution</summary>

```python
from torch.utils.data import DataLoader
import torch as _t
import numpy as _np
import random as _random

def _worker_init_fn(worker_id):
    base = _t.initial_seed() % (2 ** 32)
    _np.random.seed(base + worker_id)
    _random.seed(base + worker_id)

def ex2_make_seeded_shuffle_dataloader(dataset, batch_size, num_workers):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=num_workers,
        shuffle=True,
        pin_memory=False,
        drop_last=False,
        worker_init_fn=_worker_init_fn,
    )
```

**`torch.initial_seed()` inside a worker is the per-worker base seed PyTorch already assigned.** Calling it from a worker_init_fn lets you derive a numpy/random seed that's distinct per worker AND deterministic given `torch.manual_seed(...)` in the main process.

**`% (2**32)` is required.** numpy and Python's `random` both want a uint32 seed. `torch.initial_seed()` returns a 63-bit int; the modulo collapses it safely.

**`pin_memory=False` on CPU is correct.** `pin_memory=True` would raise at first batch fetch without CUDA. The contract we're exercising is the rest of the DataLoader config — workers, shuffle, seeded init — all of which work fine CPU-only.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()